# Step 1: Data cleaning
**The Hundred (men), 2021-2026.** Source: [Cricsheet](https://cricsheet.org), Open Data Commons Attribution License.

Rules for this project:
- One row per delivery, including wides and no-balls (they change the score).
- Anything ending `_pre` describes the match **before** the ball. Outcomes keep plain names.
- Nothing is dropped silently: every removal is counted.

In [1]:
import json, glob, os
import numpy as np, pandas as pd
ROOT = os.path.expanduser('~/hundred-momentum')
files = sorted(glob.glob(f'{ROOT}/data/raw/hnd_male/*.json'))
print('match files:', len(files))

match files: 201


## 1.1 Flatten the JSON
Super overs are skipped. A retired-hurt batter is not counted as a wicket; retired out is.

In [2]:
NOT_A_WICKET = {'retired hurt', 'retired not out'}
NOT_BOWLERS = {'run out', 'retired out', 'retired hurt', 'retired not out', 'obstructing the field', 'timed out', 'handled the ball'}
balls, matches, unavailable = [], [], []
for f in files:
    d = json.load(open(f)); i = d['info']; mid = os.path.basename(f)[:-5]; o = i['outcome']; reg = i['registry']['people']
    matches.append(dict(match_id=mid, date=i['dates'][0], season=int(str(i['season'])[:4]), match_number=i['event'].get('match_number'), stage=i['event'].get('stage'),
        venue=i.get('venue'), team1=i['teams'][0], team2=i['teams'][1], toss_winner=i['toss']['winner'], toss_decision=i['toss']['decision'],
        winner=o.get('winner') or o.get('eliminator'), result=o.get('result','win'), method=o.get('method'),
        by_runs=o.get('by',{}).get('runs'), by_wickets=o.get('by',{}).get('wickets'), n_innings=sum(not x.get('super_over') for x in d['innings'])))
    for k, inn in enumerate([x for x in d['innings'] if not x.get('super_over')], 1):
        bat = inn['team']; bowl = [t for t in i['teams'] if t != bat][0]; tg = inn.get('target', {})
        for pl in inn.get('absent_hurt', []): unavailable.append((mid, k, pl))
        for ov in inn['overs']:
            for b in ov['deliveries']:
                for w0 in b.get('wickets', []):
                    if w0['kind'] in NOT_A_WICKET: unavailable.append((mid, k, w0['player_out']))
        seq = 0
        for ov in inn['overs']:
            for b in ov['deliveries']:
                seq += 1; e = b.get('extras', {}); w = [x for x in b.get('wickets', []) if x['kind'] not in NOT_A_WICKET]
                balls.append(dict(match_id=mid, innings=k, seq=seq, over=ov['over'], batting_team=bat, bowling_team=bowl,
                    batter=b['batter'], batter_id=reg.get(b['batter']), non_striker=b['non_striker'], bowler=b['bowler'], bowler_id=reg.get(b['bowler']),
                    runs_bat=b['runs']['batter'], runs_extra=b['runs']['extras'], runs_total=b['runs']['total'],
                    wides=e.get('wides',0), noballs=e.get('noballs',0), byes=e.get('byes',0), legbyes=e.get('legbyes',0), penalty=e.get('penalty',0),
                    n_wickets=len(w), wicket_kind=w[0]['kind'] if w else None, player_out=w[0]['player_out'] if w else None,
                    bowler_wicket=int(any(x['kind'] not in NOT_BOWLERS for x in w)), target=tg.get('runs'), target_overs=tg.get('overs')))
M = pd.DataFrame(matches); M['date'] = pd.to_datetime(M['date']); B = pd.DataFrame(balls)
print(M.shape, B.shape); print(M.season.value_counts().sort_index().to_dict()); print(M.result.value_counts().to_dict(), M.method.value_counts().to_dict())

(201, 16) (38777, 25)
{2021: 32, 2022: 34, 2023: 33, 2024: 34, 2025: 34, 2026: 34}
{'win': 192, 'no result': 7, 'tie': 2} {'D/L': 2}


## 1.2 Basic integrity checks

In [3]:
print('missing IDs :', B[['batter_id','bowler_id']].isna().sum().to_dict())
print('duplicate keys:', B.duplicated(['match_id','innings','seq']).sum())
print('runs add up :', (B.runs_bat + B.runs_extra == B.runs_total).all(), '| extras add up:', (B[['wides','noballs','byes','legbyes','penalty']].sum(axis=1) == B.runs_extra).all())
print('batter runs  :', B.runs_bat.value_counts().sort_index().to_dict())
print('balls with 2+ wickets:', (B.n_wickets > 1).sum())
print('names spelt 2+ ways per ID:', int((B.groupby('batter_id').batter.nunique() > 1).sum()))
B['legal'] = ((B.wides == 0) & (B.noballs == 0)).astype(int)
print('legal balls per five, counts:', B.groupby(['match_id','innings','over']).legal.sum().value_counts().sort_index().to_dict())

missing IDs : {'batter_id': 0, 'bowler_id': 0}
duplicate keys: 0
runs add up : True | extras add up: True
batter runs  : {0: 14612, 1: 14379, 2: 2811, 3: 134, 4: 4569, 5: 5, 6: 2267}
balls with 2+ wickets: 0
names spelt 2+ ways per ID: 0
legal balls per five, counts: {1: 15, 2: 32, 3: 24, 4: 48, 5: 7388}


## 1.3 Franchise names
Teams were renamed over time. `franchise` links a team to its home ground so histories carry across seasons.

In [4]:
print(pd.crosstab(pd.concat([M.team1, M.team2], ignore_index=True), pd.concat([M.season, M.season], ignore_index=True)))
home = M.groupby('team1').venue.agg(lambda s: s.value_counts().index[0]); print(home)

season                   2021  2022  2023  2024  2025  2026
row_0                                                      
Birmingham Phoenix          9     8     7     9     8     8
London Spirit               7     9     8     8     8     8
MI London                   0     0     0     0     0     8
Manchester Originals        7    10    10     8     8     0
Manchester Super Giants     0     0     0     0     0    10
Northern Superchargers      7     8     8     8     9     0
Oval Invincibles            7     8     9     9     9     0
Southern Brave             10     8     9    10     8     8
Sunrisers Leeds             0     0     0     0     0     9
Trent Rockets               9     9     7     8    10     9
Welsh Fire                  8     8     8     8     8     8
team1
Birmingham Phoenix                          Edgbaston
London Spirit                                  Lord's
MI London                                    The Oval
Manchester Originals                     Old Traffor

In [5]:
vc = {}
for t in sorted(set(M.team1) | set(M.team2)):
    vc[t] = home.get(t)
# link teams that share a main home ground; keep the earliest name as the franchise label
first = {t: min(M[(M.team1 == t) | (M.team2 == t)].season) for t in vc}
norm = lambda v: None if v is None else v.split(',')[0].replace('The ', '').replace('Kennington ', '').strip()
F = {}
for t in sorted(vc, key=lambda x: first[x]):
    same = [u for u in F if norm(vc[u]) == norm(vc[t])]
    F[t] = F[same[0]] if same else t
print({t: f for t, f in F.items() if t != f}, '| franchises:', len(set(F.values())))
assert len(set(F.values())) == 8, 'check franchise mapping'
for col in ['batting_team', 'bowling_team']: B[col.replace('team', 'franchise')] = B[col].map(F)
for col in ['team1','team2','winner']: M[col + '_franchise'] = M[col].map(F)

{'MI London': 'Oval Invincibles', 'Manchester Super Giants': 'Manchester Originals', 'Sunrisers Leeds': 'Northern Superchargers'} | franchises: 8


## 1.4 Check totals against official results
Recompute the winner and margin from the balls. D/L, tied and no-result games are handled separately.

In [6]:
tot = B.groupby(['match_id','innings']).agg(team=('batting_team','first'), runs=('runs_total','sum'), wkts=('n_wickets','sum'), legal=('legal','sum'), target=('target','first')).reset_index()
a = tot[tot.innings == 1].set_index('match_id'); b = tot[tot.innings == 2].set_index('match_id')
chk = M.set_index('match_id').join(a[['team','runs','wkts','legal']].add_suffix('_1')).join(b[['team','runs','wkts','legal','target']].add_suffix('_2'))
std = chk[(chk.result == 'win') & chk.method.isna()]
ok_runs = std.by_runs.isna() | ((std.runs_1 - std.runs_2 == std.by_runs) & (std.winner == std.team_1))
ok_w = std.by_wickets.isna() | ((10 - std.wkts_2 == std.by_wickets) & (std.winner == std.team_2) & (std.runs_2 > std.runs_1))
ok_t = std.target_2 == std.runs_1 + 1
print('standard games:', len(std), '| runs margins ok:', int(ok_runs.sum()), '| wicket margins ok:', int(ok_w.sum()), '| targets ok:', int(ok_t.sum()))
print(std[~(ok_runs & ok_w & ok_t)][['date','team_1','runs_1','team_2','runs_2','wkts_2','target_2','by_runs','by_wickets','winner']])
print('legal balls max:', tot.legal.max(), '| innings > 100:', int((tot.legal > 100).sum()), '| wickets max:', tot.wkts.max())
print(tot[tot.legal > 100])

standard games: 190 | runs margins ok: 190 | wicket margins ok: 190 | targets ok: 190
Empty DataFrame
Columns: [date, team_1, runs_1, team_2, runs_2, wkts_2, target_2, by_runs, by_wickets, winner]
Index: []
legal balls max: 100 | innings > 100: 0 | wickets max: 10
Empty DataFrame
Columns: [match_id, innings, team, runs, wkts, legal, target]
Index: []


## 1.5 Flag shortened and unusable matches
Flagged, not deleted, so each model can choose. `full_match` = two innings, a result or tie, no D/L, and each innings ended by 100 balls, all out, or a completed chase.

In [7]:
x = chk.copy()
# a side is all out once no batters remain: absent-hurt / retired-hurt players who never resumed reduce the limit
U = pd.DataFrame(unavailable, columns=['match_id','innings','player']).drop_duplicates()
def resumed(r):
    z = B[(B.match_id == r.match_id) & (B.innings == r.innings)]; return (z.tail(1)[['batter','non_striker']] == r.player).any().any() or (z.player_out == r.player).any()
U = U[~U.apply(resumed, axis=1)]; nu = U.groupby(['match_id','innings']).size()
lim = lambda k: 10 - x.index.map(lambda q: nu.get((q, k), 0))
print('batters unavailable:', U.values.tolist())
end1 = (x.legal_1 >= 100) | (x.wkts_1 >= lim(1))
end2 = (x.legal_2 >= 100) | (x.wkts_2 >= lim(2)) | (x.runs_2 >= x.target_2)
tov = B[B.innings == 2].groupby('match_id').target_overs.first()
x['full_match'] = ((x.n_innings == 2) & x.result.isin(['win','tie']) & x.method.isna() & end1 & end2 & (tov.reindex(x.index) == 20)).astype(int)
M = M.merge(x['full_match'].reset_index(), on='match_id')
print(M.full_match.value_counts().to_dict()); print(M[M.full_match == 0][['date','team1','team2','result','method','n_innings']].join(x[['legal_1','wkts_1','legal_2','wkts_2']].reset_index(drop=True)).to_string())

batters unavailable: [['1355624', 2, 'JM Cox'], ['1417816', 2, 'BA Stokes'], ['1471014', 2, 'TP Alsop'], ['1471027', 2, 'RJW Topley']]
{1: 189, 0: 12}
          date                   team1                   team2     result method  n_innings  legal_1  wkts_1  legal_2  wkts_2
16  2021-08-05    Manchester Originals          Southern Brave  no result   None          1       71       3      NaN     NaN
19  2021-08-08        Oval Invincibles           Trent Rockets        win   None          2       65       6     65.0     8.0
67  2023-08-02              Welsh Fire    Manchester Originals        win   None          2       40       3     40.0     4.0
69  2023-08-03      Birmingham Phoenix  Northern Superchargers  no result   None          1       62       5      NaN     NaN
71  2023-08-05    Manchester Originals           London Spirit  no result   None          1       80       5      NaN     NaN
75  2023-08-08          Southern Brave           London Spirit  no result   None          1  

## 1.6 Pre-ball match state
Every `_pre` value uses earlier balls in the same innings only: cumulative total minus the current ball.

In [8]:
B = B.merge(M[['match_id','date','season','venue','full_match']], on='match_id').sort_values(['date','match_id','innings','seq']).reset_index(drop=True)
g = B.groupby(['match_id','innings'], sort=False)
B['score_pre'] = g.runs_total.cumsum() - B.runs_total
B['wkts_pre'] = g.n_wickets.cumsum() - B.n_wickets
B['legal_pre'] = g.legal.cumsum() - B.legal
B['ball_no'] = B.legal_pre + 1
B['balls_left_pre'] = 100 - B.legal_pre
B['runs_needed_pre'] = np.where(B.innings == 2, B.target - B.score_pre, np.nan)
B['is_wicket'] = (B.n_wickets > 0).astype(int)
B['phase'] = np.select([B.ball_no <= 25, B.ball_no <= 75], ['powerplay','middle'], 'death')
# independent rebuild, ball by ball, of the pre-ball score, wickets and legal balls
bad = 0
for _, z in B.groupby(['match_id','innings'], sort=False):
    s = w = l = 0
    for r, wk, lg, sp, wp, lp in z[['runs_total','n_wickets','legal','score_pre','wkts_pre','legal_pre']].itertuples(index=False):
        bad += (s != sp) or (w != wp) or (l != lp); s += r; w += wk; l += lg
print('pre-ball state mismatches:', bad, '| first ball all zero:', bool((g.head(1)[['score_pre','wkts_pre','legal_pre']] == 0).all().all()))
print('range balls_left_pre:', B.balls_left_pre.min(), B.balls_left_pre.max(), '| wkts_pre max:', B.wkts_pre.max())

pre-ball state mismatches: 0 | first ball all zero: True
range balls_left_pre: 1 100 | wkts_pre max: 9


## 1.7 Summary and save

In [9]:
L = B[(B.full_match == 1)]
print('deliveries:', len(B), '| in full matches:', len(L), '| legal:', int(L.legal.sum()), '| matches:', L.match_id.nunique())
print(L.groupby('season').agg(matches=('match_id','nunique'), balls=('seq','size'), runs_per_ball=('runs_total','mean'), wkt_pct=('is_wicket','mean')).round(3))
print(L[L.legal == 1].groupby('phase').agg(balls=('seq','size'), runs_bat=('runs_bat','mean'), wkt_pct=('is_wicket','mean'), boundary_pct=('runs_bat', lambda s: (s >= 4).mean())).round(3))
print('nulls:', {k: int(v) for k, v in B.isna().sum().items() if v})
B.to_parquet(f'{ROOT}/data/clean/balls.parquet', index=False); M.to_parquet(f'{ROOT}/data/clean/matches.parquet', index=False)
B.to_csv(f'{ROOT}/data/clean/balls.csv', index=False); M.to_csv(f'{ROOT}/data/clean/matches.csv', index=False); print('saved', B.shape, M.shape)

deliveries: 38777 | in full matches: 37680 | legal: 36235 | matches: 189
        matches  balls  runs_per_ball  wkt_pct
season                                        
2021         30   5948          1.414    0.060
2022         34   6826          1.436    0.060
2023         28   5652          1.444    0.064
2024         30   5896          1.319    0.065
2025         33   6524          1.433    0.061
2026         34   6834          1.500    0.056
           balls  runs_bat  wkt_pct  boundary_pct
phase                                            
death       8044     1.595    0.100         0.208
middle     18741     1.318    0.054         0.150
powerplay   9450     1.361    0.050         0.221


nulls: {'wicket_kind': 36400, 'player_out': 36400, 'target': 20351, 'target_overs': 20351, 'runs_needed_pre': 20351}


saved (38777, 40) (201, 20)
